# ▶️ RUN THIS NOTEBOOK IN ORDER

Run cells **1 → 4** top-to-bottom. Everything under the **📦 ARCHIVE** divider is old
exploration/debugging — you do **not** need to run it.

| # | Cell | Purpose | Required? |
|---|------|---------|-----------|
| 1 | Setup | clone repo + install deps | ✅ once |
| 2 | Core experiment | baseline vs green vs green+carbon (Qwen-0.5B × guanaco) + comparison PDFs | ✅ main energy results |
| 3 | Generalization | 6 runs: 3 models × 2 datasets + summary CSV | ⬜ optional (cross-model breadth) |
| 4 | Quality / accuracy eval | ARC/MMLU/HellaSwag on base vs baseline-LoRA vs green-LoRA | ✅ the missing piece for the paper |

> **Note:** cells 2 and 3 are *independent* — run cell 2 for the headline single-model numbers,
> run cell 3 only if you also want the multi-model/dataset table. Cell 4 works after either.

> ⚠️ **Security:** your old cells had a hard-coded **Anthropic API key** and a **GitHub token**
> (now redacted in the archive below). **Revoke/rotate both** — they are already exposed.
> Put your *own fresh* token in the Setup cell if the repo is private.

In [ ]:
# 1️⃣ SETUP — run once
# If the repo is PRIVATE, paste a FRESH GitHub token (revoke the old leaked one!).
GH_TOKEN = ""   # e.g. "ghp_xxx"; leave "" if public or already cloned
BRANCH   = "Improvement-4_carbon"

import os
auth = f"erfan38:{GH_TOKEN}@" if GH_TOKEN else ""
if not os.path.exists("/content/GreenPEFT"):
    os.system(f"git clone -b {BRANCH} https://{auth}github.com/erfan38/GreenPEFT.git /content/GreenPEFT")
%cd /content/GreenPEFT
os.environ["PYTHONPATH"] = "/content/GreenPEFT"  # so subprocesses can import energypeft
!git pull origin {BRANCH}
!pip install -q -e .
!pip install -q --upgrade "torchao>=0.16.0" "peft>=0.14.0" "transformers>=4.46.0" \
    "datasets<3.0" accelerate pynvml nvidia-ml-py psutil requests reportlab

import torchao, peft, transformers
print(f"\n  torchao={torchao.__version__}  peft={peft.__version__}  transformers={transformers.__version__}")
print("✅ Setup done. If torchao/transformers were JUST upgraded: Runtime ▸ Restart, then re-run cell 1.")

In [ ]:
# 2️⃣ CORE EXPERIMENT — baseline vs green vs green+carbon (Qwen-0.5B × guanaco)
# Produces the headline energy/CO2/loss numbers + 2 comparison PDFs.
import subprocess, glob, os

def run(cmd):
    print(f"\n{'='*60}\n$ {cmd}\n{'='*60}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode})")

BASE = (
    'python -u src/fine-tune_2.py '
    '--model "Qwen/Qwen2.5-0.5B-Instruct" '
    '--dataset "mlabonne/guanaco-llama2-1k" --split "train" '
    '--max_steps -1 --num_train_epochs 3.0 '
    '--batch 8 --grad_accum 8 --lr 2e-4 --max_length 256 '
    '--lora_r 8 --lora_alpha 32 --lora_dropout 0.1 '
    '--target_modules "q_proj,v_proj" '
    '--carbon_intensity_g_per_kwh 50 --pue 1.2 --no_codecarbon'
)
GREEN_FLAGS = (
    '--energy_budget_wh 400 --use_loss_threshold --use_early_accum_exit --convergence_threshold 0.90 --save_steps 5'
)

print("\n===== BASELINE =====", flush=True)
run(f'{BASE} --mode baseline --outroot_baseline /content/fair/baseline')
print("\n===== GREEN (no carbon) =====", flush=True)
run(f'{BASE} --mode green {GREEN_FLAGS} --outroot_green /content/fair/green_no_carbon')
print("\n===== GREEN (carbon-aware) =====", flush=True)
run(f'{BASE} --mode green {GREEN_FLAGS} --use_carbon_aware --carbon_zone "CA-QC" --outroot_green /content/fair/green_carbon')

latest_base   = sorted(glob.glob("/content/fair/baseline/*/"))[-1]
latest_green  = sorted(glob.glob("/content/fair/green_no_carbon/*/"))[-1]
latest_carbon = sorted(glob.glob("/content/fair/green_carbon/*/"))[-1]
print(f"\nBaseline: {latest_base}\nGreen:    {latest_green}\nCarbon:   {latest_carbon}")

print("\n===== COMPARE: Green vs Baseline =====", flush=True)
run(f'python -u src/compare_2.py --run_a "{latest_green}" --label_a "Green" '
    f'--run_b "{latest_base}" --label_b "Baseline" --outdir /content/fair/compare_green')
print("\n===== COMPARE: Green Carbon-Aware vs Baseline =====", flush=True)
run(f'python -u src/compare_2.py --run_a "{latest_carbon}" --label_a "Green (carbon-aware)" '
    f'--run_b "{latest_base}" --label_b "Baseline" --outdir /content/fair/compare_carbon')

try:
    from google.colab import files
    for pdf in sorted(glob.glob("/content/fair/compare_*/*.pdf")):
        print(f"Downloading: {pdf}"); files.download(pdf)
except Exception:
    print("Not in Colab — PDFs are in /content/fair/compare_*/")

In [ ]:
# 3️⃣ GENERALIZATION STUDY (OPTIONAL) — 3 models × 2 datasets, resumable
# Re-running skips already-completed runs. Set USE_LLAMA=True only after Meta approves
# your Llama-3.2 access AND you run `!hf auth login` first.
import subprocess, sys, glob, os, json, csv

USE_LLAMA = False   # False -> uses SmolLM2-1.7B (no gating)

QWEN_05B      = {"id":"Qwen/Qwen2.5-0.5B-Instruct","short":"qwen-0.5b","target_modules":"q_proj,v_proj"}
TINYLLAMA_11B = {"id":"TinyLlama/TinyLlama-1.1B-Chat-v1.0","short":"tinyllama-1.1b","target_modules":"q_proj,v_proj"}
THIRD_MODEL   = ({"id":"meta-llama/Llama-3.2-1B-Instruct","short":"llama-3.2-1b","target_modules":"q_proj,v_proj"}
                 if USE_LLAMA else
                 {"id":"HuggingFaceTB/SmolLM2-1.7B-Instruct","short":"smollm2-1.7b","target_modules":"q_proj,v_proj"})
GUANACO = {"id":"mlabonne/guanaco-llama2-1k","short":"guanaco","split":"train","needs_format":False}
DOLLY   = {"id":"databricks/databricks-dolly-15k","short":"dolly","split":"train[:1000]","needs_format":True}

RUN_PLAN = [
    (TINYLLAMA_11B, GUANACO, "baseline"), (TINYLLAMA_11B, GUANACO, "green"),
    (THIRD_MODEL,   GUANACO, "baseline"), (THIRD_MODEL,   GUANACO, "green"),
    (QWEN_05B,      DOLLY,   "baseline"), (QWEN_05B,      DOLLY,   "green"),
]
OUT_ROOT = "/content/multi_eval"; COMPARE_OUT = f"{OUT_ROOT}/comparisons"
os.makedirs(COMPARE_OUT, exist_ok=True)

def run(cmd):
    print(f"\n{'-'*60}\n$ {cmd}\n{'-'*60}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode})")

def has_completed_run(out_dir):
    for sd in sorted(glob.glob(f"{out_dir}/*/")):
        reports = glob.glob(f"{sd}/*_run_report_*.json")
        if reports:
            try:
                if json.load(open(reports[-1])).get("total_energy_wh") is not None:
                    return sd
            except Exception:
                pass
    return None

def prepare_dolly_as_text(out_path):
    if os.path.exists(out_path):
        print(f"  Dolly already prepared at {out_path}"); return
    from datasets import load_dataset
    ds = load_dataset(DOLLY["id"], split=DOLLY["split"])
    def to_text(ex):
        instr=(ex.get("instruction") or "").strip(); ctx=(ex.get("context") or "").strip(); resp=(ex.get("response") or "").strip()
        text=(f"### Instruction:\n{instr}\n\n### Context:\n{ctx}\n\n### Response:\n{resp}" if ctx
              else f"### Instruction:\n{instr}\n\n### Response:\n{resp}")
        return {"text": text}
    ds.map(to_text, remove_columns=ds.column_names).save_to_disk(out_path)

def build_command(model, dataset, mode, out_dir):
    base = (f'python -u src/fine-tune_2.py --model "{model["id"]}" '
            f'--max_steps -1 --num_train_epochs 3.0 --batch 8 --grad_accum 8 --lr 2e-4 --max_length 256 '
            f'--lora_r 8 --lora_alpha 32 --lora_dropout 0.1 --target_modules "{model["target_modules"]}" '
            f'--carbon_intensity_g_per_kwh 50 --pue 1.2 --no_codecarbon')
    if dataset["needs_format"]:
        base += f' --dataset "/content/datasets_local/{dataset["short"]}_text" --split "train"'
    else:
        base += f' --dataset "{dataset["id"]}" --split "{dataset["split"]}"'
    if mode == "baseline":
        return f'{base} --mode baseline --outroot_baseline "{out_dir}"'
    return (f'{base} --mode green --energy_budget_wh 400 --use_loss_threshold '
            f'--use_early_accum_exit --convergence_threshold 0.90 --outroot_green "{out_dir}"')

os.makedirs("/content/datasets_local", exist_ok=True)
prepare_dolly_as_text(f"/content/datasets_local/{DOLLY['short']}_text")

run_dirs = {}
for i, (model, dataset, mode) in enumerate(RUN_PLAN, 1):
    key = (model["short"], dataset["short"], mode)
    out_dir = f"{OUT_ROOT}/{model['short']}/{dataset['short']}/{mode}"; os.makedirs(out_dir, exist_ok=True)
    print(f"\n  Run {i}/{len(RUN_PLAN)}: {model['short']} x {dataset['short']} x {mode}")
    done = has_completed_run(out_dir)
    if done:
        print(f"  SKIP (already done): {done}"); run_dirs[key] = done; continue
    try:
        run(f'cd /content/GreenPEFT && {build_command(model, dataset, mode, out_dir)}')
    except Exception as e:
        print(f"  !! failed: {e}"); continue
    subs = sorted(glob.glob(f"{out_dir}/*/"))
    if subs: run_dirs[key] = subs[-1]

for model, dataset in [(TINYLLAMA_11B, GUANACO), (THIRD_MODEL, GUANACO), (QWEN_05B, DOLLY)]:
    bk=(model["short"],dataset["short"],"baseline"); gk=(model["short"],dataset["short"],"green")
    if bk not in run_dirs or gk not in run_dirs:
        print(f"  skip compare {model['short']} x {dataset['short']}"); continue
    outdir=f"{COMPARE_OUT}/{model['short']}_{dataset['short']}"; os.makedirs(outdir, exist_ok=True)
    run(f'cd /content/GreenPEFT && python -u src/compare_2.py '
        f'--run_a "{run_dirs[gk]}" --label_a "Green ({model["short"]})" '
        f'--run_b "{run_dirs[bk]}" --label_b "Baseline ({model["short"]})" --outdir "{outdir}"')

rows=[]
for (m,d,mode),rd in run_dirs.items():
    reps=sorted(glob.glob(f"{rd}/*_run_report_*.json"))
    if not reps: continue
    r=json.load(open(reps[-1]))
    rows.append({"model":m,"dataset":d,"mode":mode,"energy_wh":round(r.get("total_energy_wh",0),4),
                 "time_seconds":round(r.get("time_seconds_wall",0),1),"co2_kg_pue":r.get("co2_kg_operational_x_pue",0),
                 "skip_rate_pct":r.get("skip_rate_pct",0),"run_dir":rd})
csv_path=f"{COMPARE_OUT}/summary.csv"
if rows:
    with open(csv_path,"w",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    for row in rows:
        print(f"  {row['model']:16s} {row['dataset']:9s} {row['mode']:9s} E={row['energy_wh']:.3f}  T={row['time_seconds']:.0f}s  skip={row['skip_rate_pct']:.1f}%")
try:
    from google.colab import files
    for pdf in sorted(glob.glob(f"{COMPARE_OUT}/*/*.pdf")): files.download(pdf)
    if os.path.exists(csv_path): files.download(csv_path)
except Exception:
    print(f"Not in Colab — outputs in {COMPARE_OUT}/")
print(f"\n✅ Done — {len(run_dirs)}/{len(RUN_PLAN)} runs")

In [ ]:
# 4️⃣ QUALITY / ACCURACY EVALUATION — the missing piece for the paper
# Evaluates base (no FT) vs baseline-LoRA vs green-LoRA on standard benchmarks.
# Backend: EleutherAI lm-evaluation-harness (deterministic, NO API key needed).
!pip install -q "lm-eval>=0.4.5" peft accelerate pandas

import glob, os, datetime
import pandas as pd
from lm_eval import simple_evaluate

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# The trainer saves the LoRA adapter inside a checkpoint-* subfolder, so search recursively
# for the folder that actually contains adapter_config.json.
def find_adapter(run_glob):
    runs = sorted(glob.glob(run_glob))
    if not runs:
        return None
    hits = glob.glob(os.path.join(runs[-1], "**", "adapter_config.json"), recursive=True)
    return os.path.dirname(sorted(hits)[-1]) if hits else None

# auto-detect adapters from cell 2 (falls back to cell 3 / older dirs); EDIT if needed
BASELINE_ADAPTER = find_adapter("/content/fair/baseline/*/") or find_adapter("/content/baseline_results/*/")
GREEN_ADAPTER    = (find_adapter("/content/fair/green_carbon/*/") or find_adapter("/content/fair/green_no_carbon/*/")
                    or find_adapter("/content/green_results_2/*/"))
print("BASELINE adapter:", BASELINE_ADAPTER)
print("GREEN    adapter:", GREEN_ADAPTER)
assert BASELINE_ADAPTER and GREEN_ADAPTER, "Run cell 2 (core experiment) first, or set paths manually."

TASKS       = ["arc_easy", "arc_challenge", "hellaswag", "winogrande"]   # add "mmlu" for final
NUM_FEWSHOT = 0       # set 25 (ARC) / 5 (MMLU) for paper numbers
LIMIT       = 200     # quick smoke test; set None for full benchmark

def evaluate(label, adapter=None):
    margs = f"pretrained={BASE_MODEL},dtype=bfloat16"
    if adapter:
        margs += f",peft={adapter}"
    res = simple_evaluate(model="hf", model_args=margs, tasks=TASKS,
                          num_fewshot=NUM_FEWSHOT, batch_size="auto", limit=LIMIT)
    row = {"model": label}
    for t, m in res["results"].items():
        acc = m.get("acc_norm,none", m.get("acc,none"))
        row[t] = round(100 * acc, 2) if acc is not None else None
    return row

rows = [evaluate("base (no FT)"),
        evaluate("baseline-LoRA", BASELINE_ADAPTER),
        evaluate("green-LoRA",    GREEN_ADAPTER)]
df = pd.DataFrame(rows).set_index("model"); df["avg"] = df.mean(axis=1).round(2)
print("\n=== Downstream accuracy (%) — higher is better ===")
print(df.to_string())
print("\nKey check: green-LoRA 'avg' within ~1 pt of baseline-LoRA 'avg' = quality preserved.")

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S"); out = f"/content/quality_eval_{ts}.csv"
df.to_csv(out); print("saved:", out)
try:
    from google.colab import files; files.download(out)
except Exception:
    pass